# Multi-Hop Retrieval — Train FakeEncoder (retrieval_v2)

**Before running:**
1. Settings → Accelerator → **T4 GPU**
2. Settings → Internet → **On**
3. Google Drive `musique_data/` folder must contain:
   - `musique_ans_v1.0_train.jsonl`
   - `musique_ans_v1.0_dev.jsonl`

**What this trains:**
- `FakeEncoderModel` — Encoder1 (BERT-base) + FakeEncoder (3-layer Transformer) + Decoder (auto-regressive, teacher-forced)
- **Loss:** CrossEntropy on B generation — FakeEncoder must supply what Encoder1 cannot decode from A alone
- **F update:** Reverse cross-attention + GRUCell — F refines over N_ITER=3 passes per example
- **Complement:** mean_pool(FakeEncoder(final_F)) → proj → L2-norm → 128-dim
- **Dataset:** ALL 26,675 hop (A,B) pairs from MuSiQue training set

**Validation metric:** reconstruction loss (cross-entropy on held-out val B texts)
- Epoch 1: BERT frozen, only FakeEncoder + Decoder train (~30 min on T4)
- Epoch 2-3: BERT unfrozen at lr × 0.1 (~40 min per epoch)

**Expected time: ~2 hours on T4**

**After training:** download `fakencoder_best.pt` from Output tab → upload to Google Drive `musique_data/`

In [ ]:
# ── [EDIT IF NEEDED] ───────────────────────────────────────────────────────────
REPO_URL        = "https://github.com/haaaaaaayrewugrhyw/multihop-retrieval.git"
REPO_NAME       = "multihop-retrieval"
DRIVE_FOLDER_ID = "1mMCf_lhYcw3CH_ttOWDLgSKZuPYqh5m5"   # musique_data folder ID
WORK_DIR        = f"/kaggle/working/{REPO_NAME}/retrieval_v2"
# ───────────────────────────────────────────────────────────────────────────────

In [ ]:
# 1. Verify GPU — must be T4 (sm_70+), not P100 (sm_60)
import torch

if not torch.cuda.is_available():
    raise RuntimeError("No GPU detected. Settings → Accelerator → T4 GPU")

props   = torch.cuda.get_device_properties(0)
cc      = props.major
vram_gb = props.total_memory / 1e9

print(f"GPU  : {torch.cuda.get_device_name(0)}")
print(f"VRAM : {vram_gb:.1f} GB")
print(f"SM   : {cc}.{props.minor}")

if cc < 7:
    raise RuntimeError(
        f"GPU is P100 (sm_{cc}0) — not supported by PyTorch 2.x.\n"
        "FIX: Settings → Accelerator → change to T4 GPU"
    )
print("CUDA OK")

In [ ]:
# 2. Clone repo + install dependencies
import os

repo_root = f"/kaggle/working/{REPO_NAME}"
if not os.path.isdir(repo_root):
    os.system(f"git clone {REPO_URL} {repo_root}")
else:
    os.system(f"cd {repo_root} && git pull")

os.chdir(WORK_DIR)
print("Working dir:", os.getcwd())

os.system("pip install -q 'transformers>=4.35.0' 'rank_bm25>=0.2.2' gdown")
print("Dependencies ready")

In [ ]:
# 3. Download MuSiQue data from Google Drive
import os, gdown

download_dir = "/kaggle/working/drive_data"

if not os.path.isdir(download_dir):
    print("Downloading from Google Drive...")
    gdown.download_folder(
        id=DRIVE_FOLDER_ID,
        output=download_dir,
        quiet=False,
        use_cookies=False,
    )
else:
    print("Drive data already downloaded")

print("\nDownloaded files:")
for f in sorted(os.listdir(download_dir)):
    size = os.path.getsize(f"{download_dir}/{f}") / 1e6
    print(f"  {f:45s}  {size:7.1f} MB")

In [ ]:
# 4. Set up expected file paths
import os

drive = "/kaggle/working/drive_data"

os.makedirs(f"{WORK_DIR}/data/musique", exist_ok=True)
os.makedirs(f"{WORK_DIR}/models",       exist_ok=True)
os.makedirs(f"{WORK_DIR}/cache",        exist_ok=True)
os.makedirs(f"{WORK_DIR}/results",      exist_ok=True)

for fname in ["musique_ans_v1.0_train.jsonl", "musique_ans_v1.0_dev.jsonl"]:
    src = f"{drive}/{fname}"
    dst = f"{WORK_DIR}/data/musique/{fname}"
    if not os.path.exists(dst):
        os.symlink(src, dst)
    print(f"  {fname}: OK ({os.path.getsize(src)/1e6:.0f} MB)")

print("\nAll paths ready")

In [ ]:
# 5. Smoke test — verify data loader and model init
import os
os.chdir(WORK_DIR)

print("=== Smoke test (50 examples, CPU) ===")
os.system("python fakencoder_train.py --smoke")

---
## Train FakeEncoder — Full 3-epoch run

**Architecture:**
- Encoder1 (BERT-base): `A_text → [n_A × 768]` hidden states (K1, V1)
- FakeEncoder (3-layer Transformer): `F=[8×768] zeros` → K2, V2
- Decoder (2-layer, teacher-forced): dual cross-attention + meta-attention merge → predict B
- F update: reverse cross-attention + GRUCell (gated, stable)

**Epoch schedule:**
- Epoch 1: BERT frozen — FakeEncoder + Decoder warm up
- Epoch 2-3: BERT unfrozen at lr × 0.1

In [ ]:
# 6. Train FakeEncoder (full 3-epoch run)
import os
os.chdir(WORK_DIR)

log_file = "/kaggle/working/fakencoder_train.log"
os.system(f"python fakencoder_train.py 2>&1 | tee {log_file}")
print("\nTraining complete")

In [ ]:
# 7. Verify checkpoints and copy to /kaggle/working/ root for download
import os, shutil

best  = f"{WORK_DIR}/models/fakencoder_best.pt"
final = f"{WORK_DIR}/models/fakencoder_final.pt"

for path in [best, final]:
    if os.path.exists(path):
        print(f"  {os.path.basename(path)}: {os.path.getsize(path)/1e6:.1f} MB  OK")
    else:
        print(f"  {os.path.basename(path)}: NOT FOUND — check training log")

# Copy best checkpoint to /kaggle/working/ root — appears in Output tab
if os.path.exists(best):
    out_path = "/kaggle/working/fakencoder_best.pt"
    shutil.copy(best, out_path)
    print(f"\n  Copied to {out_path}  ({os.path.getsize(out_path)/1e6:.1f} MB)")
    print("  → Download from the Output tab and upload to Drive musique_data/ folder")

print("\n--- Last 40 lines of training log ---")
with open("/kaggle/working/fakencoder_train.log") as f:
    lines = f.readlines()
print("".join(lines[-40:]))

---
## Done

`fakencoder_best.pt` is in `/kaggle/working/` — visible in the **Output tab** on the right.

**To save to Google Drive:**
1. Output tab → find `fakencoder_best.pt` → click download
2. Upload it to your Google Drive `musique_data/` folder

**Next step:** Run `eval_kaggle.ipynb` (in retrieval_v2/) to evaluate retrieval R@10.